## <b> MCP (Model Context Protocol)

In [1]:
# required Libraries
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport, StdioTransport


We are going to connect Context7 MCP server via two different transport method STDIO and HTTP.

<b>STDIO Transport

In [2]:
stdio_transport = StdioTransport(
    command = "npx",
    args=["-y", "@upstash/context7-mcp"]
)

In [3]:
stdio_client = Client(stdio_transport)

In [ ]:
# Most MCP stdio clients do not work properly in jupyter notebooks

async with stdio_client as client:
    # List of tools the server provides
    tools = await client.list_tools()

print("Done")

In [ ]:
print(len(tools))
print(
f""" name: {tools[1].name}: \n
description: {tools[1].description} \n
inputSchema: {tools[1].inputSchema}""")

<b>Call Tool

call_tool is the method we use to actually run/execute a specific tool on the MCP server

In [7]:
async with stdio_client as client:
    response = await client.call_tool("resolve-library-id",{
        "libraryName": "fastmcp",
        "query":"I want to create a new MCP server using the fastmcp python framework"}

    )
print(response.content[0].text)

RuntimeError: Client failed to connect: fileno

<b>Query Docs


query-docs tool to retrieve the actual documentation for that specific library

In [ ]:
async with stdio_client as client:
    # Use resolved ID to fetch documentation
    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

    print(docs.content[0].text[:1000])

#### HTTP Preface

In [8]:
import requests

In [9]:
url='https://www.ibm.com/'
r=requests.get(url)

In [10]:
r.status_code

200

In [11]:
print(r.request.headers)
print("request body:", r.request.body)

{'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate, br', 'Accept': '*/*', 'Connection': 'keep-alive', 'Cookie': '_abck=4704A1D90251237FCB78811049D4D4D7~-1~YAAQPvV0aAgLrE2fAQAArZoDhhAQgduTLhqYyCa8NWLjU+1AptfrDesEoEKaiVdt8St47nXNhtOlKDGzbjfrbi1nAVnBVJvnfu0PlXMd1EXty4XdrrZEeUsm5YzrphHFJEdeFULK2/D6jf1X4WuhO9SOUHCS/+ADQm1r6jDW13MZwvEupo4+/mYvBGNxtmB6gD3oiedvjBny5l6kqxs6uueuuVZ86hJccxYAwE24Kne9I4EWpzzm6dkrNrvphXfcKjZp/Mf6nwqlNApJgtjsoAWS5zZXOGRKreurBkIbDoSRL6JwHNTRJ6E0Dhq8JFmLaRw6x49KOGwsZdG4O1vWNy/aW/cojEnAj79FpbuaDNPU5rhyF8NgrxWanfXfPO69fVHc/vMulShe9lJBKLKQpDiqmU5FfTJGnZPZ36J0X34xZzvI0f0vURD8DwGSh2DRQbQ=~-1~-1~-1~-1~-1; bm_sz=3289448677CFBF2F672343E999899C48~YAAQPvV0aAkLrE2fAQAArZoDhgDM9l+9kcY1kFJ3oz9Q7nRFgyyx31OD8Sgk9kNB1Ph+jADNzQ+yvfpWhzxfyhwlTvDgvOlE19PF7zA/Qi3JeRBH/2UCdlehAF51D/oAtVBCXIcPQHekC9wufCWIjEYqdNH5+Ks9tk3/jVHh/OyENCReMPUw9lQuRExyshYLsiHlmMfQZRh+mOhxSzQJKKTg3tX4BWKkqZHfbpjZfmrZromMwiloB1Qci8IQUpTq1n6SjcJWSkw0vjMEn++jr2aSvkUV2cRfEZ6lMgLvammp8F5AIy00oH9Hk

In [12]:
# creating the streamablehttptransport
http_transport = StreamableHttpTransport(
    url="https://mcp.context7.com/mcp"
)

In [13]:
http_client = Client(http_transport)

In [14]:
async with http_client as client:
    tools = await client.list_tools()

    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

In [15]:
for tool in tools:
        print(
f"""{tool.name}: \n
{tool.description} \n
{tool.inputSchema}""")
print(response.content[0].text[:1000])
print(docs.content[0].text[:500]) 

resolve-library-id: 

Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'Query Documentation' tool to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Each result includes:
- Library ID: Context7-compatible identifier (format: /org/project)
- Name: Library or package name
- Description: Short summary
- Code Snippets: Number of available code examples
- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)
- Benchmark Score: Quality indicator (100 is the highest score)
- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.

For best results, select libraries based on name match, source reputation, snippet coverage, benchmark score, and relevance to your use ca